In [6]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import time
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import scipy.io as sio


def cseis():
    colors = [
        (0.0, 0.0, 0.0),   
        (0.2, 0.2, 0.5),
        (0.4, 0.4, 0.8),
        (0.6, 0.6, 1.0),
        (0.8, 0.8, 1.0),
        (1.0, 1.0, 1.0),    
        (1.0, 0.8, 0.8),
        (1.0, 0.6, 0.6),
        (1.0, 0.4, 0.4),
        (0.8, 0.2, 0.2),
        (0.6, 0.0, 0.0)     
    ]
    return LinearSegmentedColormap.from_list('cseis', colors, N=256)

def get_patch_position(patch_idx, data_shape, patch_size=25, stride=1):
    H, W = data_shape
    ph, pw = patch_size, patch_size
    sh, sw = stride, stride
    num_patches_per_row = (W - pw) // sw + 1
    patch_row = patch_idx // num_patches_per_row
    patch_col = patch_idx % num_patches_per_row
    top = patch_row * sh
    left = patch_col * sw
    return top, left, top + ph, left + pw

def plot_patch_on_seismic(Dn, patch_idx, patch_size=25, save_path="patch_position.png"):
    top, left, bottom, right = get_patch_position(patch_idx, Dn.shape, patch_size)
    
    H, W = Dn.shape

    ==
    plt.figure(figsize=(1, 1 * H / W))  
    
    ax = plt.gca()
    ax.imshow(Dn, cmap=cseis(), aspect='auto')
    
    rect = plt.Rectangle((left, top), patch_size, patch_size, 
                         linewidth=1, edgecolor='red', facecolor='none')
    ax.add_patch(rect)
    
    plt.title(f"The position of the {patch_idx}-th patch", fontsize=8)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\n📍 Info for patch #{patch_idx}: ")
    print(f"    Original image shape: {Dn.shape}")
    print(f"    Top-left corner: row {top}, column {left}")
    print(f"    Bottom-right corner: row {bottom}, column {right}")
    print(f"    Patch size: {patch_size}×{patch_size}\n")

In [2]:
def patch2d(A, l1=25, l2=25, s1=1, s2=1, mode=1):
    [n1, n2] = A.shape
    if mode == 1:
        tmp = np.mod(n1 - l1, s1)
        if tmp != 0:
            A = np.concatenate((A, np.zeros([s1 - tmp, n2])), axis=0)
        tmp = np.mod(n2 - l2, s2)
        if tmp != 0:
            A = np.concatenate((A, np.zeros([A.shape[0], s2 - tmp])), axis=1)
        [N1, N2] = A.shape
        X = []
        for i1 in range(0, N1 - l1 + 1, s1):
            for i2 in range(0, N2 - l2 + 1, s2):
                tmp = np.reshape(A[i1:i1+l1, i2:i2+l2], (l1*l2, 1), order='F')
                X.append(tmp)
        X = np.array(X)
    else:
        pass
    return X[:, :, 0]


In [3]:
# --- FCB ---
class FCB(nn.Module):
    def __init__(self, input_size, output_size, dropout=0.1):
        super().__init__()
        self.fc = nn.Linear(input_size, output_size)
        self.activation = nn.LeakyReLU(0.1, inplace=True)
        self.norm = nn.LayerNorm(output_size)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        x = self.fc(x)
        x = self.activation(x)
        x = self.norm(x)
        x = self.dropout(x)
        return x

# --- SparseAttention---
class SparseAttention(nn.Module):
    def __init__(self, dim, num_heads=4, window_size=2):
        super().__init__()
        assert dim % num_heads == 0, "dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.window_size = window_size
        
        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        self.proj = nn.Linear(dim, dim)
    
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        
        
        block_size = self.window_size * 2
        num_blocks = max(1, N // block_size)
        mask = torch.zeros(N, N, device=x.device)
        for i in range(num_blocks):
            start = i * block_size
            end = min(start + block_size, N)
            mask[start:end, start:end] = 1
        mask = mask.unsqueeze(0).unsqueeze(0).bool()
        
        attn = attn.masked_fill(~mask, -1e9)
        attn = attn.softmax(dim=-1)
        
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x

# --- Sparse_PatchAE ---
class Sparse_PatchAE(nn.Module):
    def __init__(self, input_dim, embed_dim=256, depth=1, num_heads=4, window_size=2):
        super().__init__()
        self.input_dim = input_dim
        self.embed_dim = embed_dim
        
        self.embed = nn.Linear(input_dim, embed_dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, 2, embed_dim) * 0.02)
        
        self.num_heads = num_heads
        self.window_size = window_size
        
        self.blocks = nn.ModuleList([
            self._make_block(embed_dim) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.decoder = nn.Linear(embed_dim, input_dim)
    
    def _make_block(self, dim):
        return nn.ModuleList([
            nn.LayerNorm(dim),
            SparseAttention(dim, self.num_heads, self.window_size),
            nn.LayerNorm(dim),
            nn.Sequential(
                nn.Linear(dim, dim * 4),
                nn.GELU(),
                nn.Linear(dim * 4, dim)
            )
        ])
    
    def forward(self, x):
        B = x.shape[0]
        x = self.embed(x).unsqueeze(1)
        
        # cls token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        
    
        for norm1, attn, norm2, mlp in self.blocks:
            x = x + attn(norm1(x))
            x = x + mlp(norm2(x))
        
        x = self.norm(x)
        patch_token = x[:, 1, :]
        recon = self.decoder(patch_token)
        return recon

# --- Encoder ---
class Encoder(nn.Module):
    def __init__(self, input_size, dropout=0.1):
        super().__init__()
        self.fcb1 = FCB(input_size, 512, dropout)
        self.Sparse1 = Sparse_PatchAE(512, 512, depth=4)
        
        self.fcb2 = FCB(512, 128, dropout)
        self.Sparse2 = Sparse_PatchAE(128, 128, depth=4)
        
        self.fcb3 = FCB(128, 64, dropout)
        self.Sparse3 = Sparse_PatchAE(64, 64, depth=4)
        
        self.fcb4 = FCB(64, 32, dropout)
        self.Sparse4 = Sparse_PatchAE(32, 32, depth=4)
        
        self.fcb5 = FCB(32, 8, dropout)
        self.Sparse5 = Sparse_PatchAE(8, 8, depth=4)
    
    def forward(self, x):
        x1 = self.fcb1(x)
        y1 = self.Sparse1(x1)
        x2 = self.fcb2(x1)
        y2 = self.Sparse2(x2)
        x3 = self.fcb3(x2)
        y3 = self.Sparse3(x3)
        x4 = self.fcb4(x3)
        y4 = self.Sparse4(x4)
        x5 = self.fcb5(x4)
        y5 = self.Sparse5(x5)
        return x1, x2, x3, x4, x5, y1, y2, y3, y4, y5

# --- Decoder ---
class Decoder(nn.Module):
    def __init__(self, output_size, dropout=0.1):
        super().__init__()
        self.fcb5 = FCB(8 + 8, 32, dropout)
        self.fcb4 = FCB(32 + 32, 64, dropout)
        self.fcb3 = FCB(64 + 64, 128, dropout)
        self.fcb2 = FCB(128 + 128, 512, dropout)
        # self.fcb1 = FCB(512 + 512, output_size, dropout)
        self.fcb1 = nn.Linear(512 + 512, output_size)
    
    def forward(self, x1, x2, x3, x4, x5, y1, y2, y3, y4, y5):
        y51 = torch.cat([x5, y5], dim=1)
        y41 = self.fcb5(y51)
        y41 = torch.cat([y41, y4], dim=1)
        y31 = self.fcb4(y41)
        y31 = torch.cat([y31, y3], dim=1)
        y21 = self.fcb3(y31)
        y21 = torch.cat([y21, y2], dim=1)
        y11 = self.fcb2(y21)
        y11 = torch.cat([y11, y1], dim=1)
        output = self.fcb1(y11)
        return output

# --- AutoCoder ---
class AutoCoder(nn.Module):
    def __init__(self, input_size, output_size, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(input_size, dropout)
        self.decoder = Decoder(output_size, dropout)
    
    def forward(self, x):
        x1, x2, x3, x4, x5, y1, y2, y3, y4, y5 = self.encoder(x)
        output = self.decoder(x1, x2, x3, x4, x5, y1, y2, y3, y4, y5)
        return output

In [4]:
class FeatureExtractor:
    """Extract intermediate layer features using hooks"""
    def __init__(self, model, layer_names):
        self.model = model
        self.layer_names = layer_names
        self.features = {}
        self.handles = []
        
        # Register hooks
        for name, module in model.named_modules():
            if name in layer_names:
                handle = module.register_forward_hook(self.get_hook(name))
                self.handles.append(handle)
    
    def get_hook(self, name):
        def hook(module, input, output):
            # Process the output of SparseAttention (take only the first element if it is a tuple)
            if isinstance(output, tuple):
                self.features[name] = output[0].detach().cpu()
            else:
                self.features[name] = output.detach().cpu()
        return hook
    
    def __call__(self, x):
        self.features = {}
        with torch.no_grad():  
            return self.model(x)
    
    def remove_hooks(self):
        for handle in self.handles:
            handle.remove()


def visualize_seismic_network_layers(model, input_patch, layer_names=None, save_path=None, clean_patch=None, mat_save_path="network_features new.mat.mat"):
    
    model.eval()
    device = next(model.parameters()).device
    input_patch = input_patch.to(device)

    if layer_names is None:
        layer_names = [
            # encoder
            'encoder.fcb1.activation',
            'encoder.Sparse1',
            'encoder.fcb2.activation',
            'encoder.Sparse2',
            'encoder.fcb3.activation',
            'encoder.Sparse3',
            'encoder.fcb4.activation',
            'encoder.Sparse4',
            'encoder.fcb5.activation',
            'encoder.Sparse5',
            # decoder
            'decoder.fcb5.activation',
            'decoder.fcb4.activation',
            'decoder.fcb3.activation',
            'decoder.fcb2.activation',
            'decoder.fcb1'
        ]

    extractor = FeatureExtractor(model, layer_names)
    output_patch = extractor(input_patch)

    n_layers = len(layer_names)
    n_cols = 4
    n_rows = (n_layers + n_cols - 1) // n_cols + 1

    fig = plt.figure(figsize=(20, 5 * n_rows))
    plt.rcParams['font.sans-serif'] = ['SimHei']
    plt.rcParams['axes.unicode_minus'] = False

    mat_data = {}

    # 1.input
    ax = plt.subplot(n_rows, n_cols, 1)
    input_img = input_patch[0].cpu().numpy().reshape(25, 25)
    mat_data["input_noisy_data"] = input_img
    im = ax.imshow(input_img, cmap=cseis(), aspect='auto')
    ax.set_title('Input noisy data (25×25)', fontsize=12)
    ax.axis('off')
    plt.colorbar(im, ax=ax, shrink=0.8)

    current_col = 2
    # 2.Clean
    if clean_patch is not None:
        clean_patch = clean_patch.to(device)
        ax = plt.subplot(n_rows, n_cols, current_col)
        clean_img = clean_patch[0].cpu().numpy().reshape(25, 25)
        mat_data["clean_reference_data"] = clean_img
        im = ax.imshow(clean_img, cmap=cseis(), aspect='auto')
        ax.set_title('CleanData)', fontsize=12)
        ax.axis('off')
        plt.colorbar(im, ax=ax, shrink=0.8)
        current_col += 1

    # 3.output
    ax = plt.subplot(n_rows, n_cols, current_col)
    output_img = output_patch[0].cpu().numpy().reshape(25, 25)
    mat_data["model_output_denoised"] = output_img
    im = ax.imshow(output_img, cmap=cseis(), aspect='auto')
    ax.set_title('output', fontsize=12)
    ax.axis('off')
    plt.colorbar(im, ax=ax, shrink=0.8)

    
    for i, name in enumerate(layer_names):
        if name not in extractor.features:
            continue

        features = extractor.features[name]

        if features.dim() == 2:
            feat = features[0]
        elif features.dim() == 3:
            feat = features[0].flatten()
        else:
            feat = features.flatten()

        feat_np = feat.cpu().numpy()
        feat_dim = feat_np.size

        # ==============================================
        # All layers automatically match perfect rectangles
        # ==============================================
        if feat_dim == 512:
            feat_img = feat_np.reshape(16, 32)
        elif feat_dim == 128:
            feat_img = feat_np.reshape(8, 16)
        elif feat_dim == 64:
            feat_img = feat_np.reshape(8, 8)
        elif feat_dim == 32:
            feat_img = feat_np.reshape(4, 8)
        elif feat_dim == 8:
            feat_img = feat_np.reshape(2, 4)
        elif feat_dim == 625:
            feat_img = feat_np.reshape(25, 25)
        else:

            feat_img = feat_np.reshape(1, feat_dim)

        # Normalization
        feat_img_normalized = (feat_img - feat_img.min()) / (feat_img.max() - feat_img.min() + 1e-8)

        var_name = name.replace(".", "_")
        mat_data[var_name] = feat_img_normalized

        ax = plt.subplot(n_rows, n_cols, i + n_cols + 1)
        im = ax.imshow(feat_img_normalized, cmap='viridis', aspect='auto')
        ax.set_title(f'{name}\nReal dimensions / Actual dimensions: {feat_dim}', fontsize=10)
        ax.axis('off')
        plt.colorbar(im, ax=ax, shrink=0.6)

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"✅ Visualization images have been saved: {save_path}")

    sio.savemat(mat_save_path, mat_data)
    print(f"✅ All real features have been saved: {mat_save_path}")

    extractor.remove_hooks()
    return extractor.features

In [5]:
# ================= 1. Loading model =================
print("Loading model...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device: {device}")

# Note: input_size and output_size are 2525=625
model = AutoCoder(input_size=25*25, output_size=25*25, dropout=0.1).to(device)

# Loading trained weights
model.load_state_dict(torch.load('syn2d_ISANET_denoise.pth', map_location=device))

model.eval()  #Switching to evaluation mode
print("✅ Model loaded successfully and switched to eval mode")

# ================= 2. Loading data =================
print("\nLoading data...")
data = sio.loadmat('seismic_snr_-4dB.mat')

# Extracting variables
Dn = data['DataNoisy']    
Dc = data['DataClean']
# Extracting 25×25 patches (parameters are fully consistent with training)
w1 = 25
w2 = 25
s1 = 1
s2 = 1

print("Extracting patch...")
X_noisy = patch2d(Dn, w1, w2, s1, s2)
X_clean = patch2d(Dc, w1, w2, s1, s2)
print(f"✅ Data loading completed")
print(f"    - Noisy patches: {X_noisy.shape}, dimension per patch: {X_noisy.shape[1]}")
print(f"    - Clean patches: {X_clean.shape}, dimension per patch: {X_clean.shape[1]}")

正在加载模型...
使用设备: cpu


FileNotFoundError: [Errno 2] No such file or directory: 'syn2d_ISANET_denoise.pth'

In [ ]:
# ================= Select the patch to visualize =================
# You can modify the number here to choose different patches
patch_idx = 8000  # 

print(f"正在可视化第 {patch_idx} 个patch...")

# Extract the patch
noisy_patch = X_noisy[patch_idx:patch_idx+1]  # shape(1, 625)
clean_patch = X_clean[patch_idx:patch_idx+1] 
# Convert to PyTorch tensors
noisy_tensor = torch.from_numpy(noisy_patch.astype(np.float32))
clean_tensor = torch.from_numpy(clean_patch.astype(np.float32))
# ================= Run visualization =================
# This will generate a large image showing input, output, and all intermediate layers
features = visualize_seismic_network_layers(
    model=model,
    input_patch=noisy_tensor,
    clean_patch=clean_tensor,
    save_path=f'layer_visualization_patch{patch_idx}_25x25 wusvit.png'  # Save the image
)

plot_patch_on_seismic(Dn, patch_idx, patch_size=25)
print("Visualization completed!")

In [ ]:
import numpy as np
import scipy.io as sio

# ===================== Load data =====================
mat_path = "network_features wusvit.mat"
data = sio.loadmat(mat_path)

# Only take sparse1~sparse5
feat_names = [
    "encoder_Sparse1",
    "encoder_Sparse2",
    "encoder_Sparse3",
    "encoder_Sparse4",
    "encoder_Sparse5"
]

# =====================  Global unified color scale  =====================
all_feats = [data[name] for name in feat_names]
global_vmin = min([np.min(f) for f in all_feats])
global_vmax = max([np.max(f) for f in all_feats])

# ===================== Calculate and print feature variance  =====================
print("="*60)
print("          Feature variance")
print("="*60)

for i, name in enumerate(feat_names):
    feat = data[name]
    # Calculate variance of the entire map
    var = np.var(feat)
    print(f"{name:20s} | variance = {var:.8f}")

print("="*60)